In [17]:
# !pip install gradio

In [18]:
import random
import gradio as gr
import matplotlib.pyplot as plt
import io
from PIL import Image

In [ ]:
class Player:
    def __init__(self, name):
        self.name = name
        self.score = 0
        self.history = []
        self.score_progression = [0]

    def record_move(self, move, points):
        self.history.append(move)
        self.score += points
        self.score_progression.append(self.score)

class AlwaysCooperate(Player):
    def make_move(self, opp_history): return "Cooperate"

class AlwaysBetray(Player):
    def make_move(self, opp_history): return "Betray"

class TitForTat(Player):
    def make_move(self, opp_history):
        return opp_history[-1] if opp_history else "Cooperate"

In [ ]:
class SimulationEngine:
    @staticmethod
    def get_scores(m1, m2): # Score Matrix
        if m1 == "Cooperate" and m2 == "Cooperate":
            return 3, 3
        if m1 == "Betray" and m2 == "Cooperate":
            return 5, 0
        if m1 == "Cooperate" and m2 == "Betray":
            return 0, 5
        return 1, 1

    def run_sim(self, strat1, strat2, rounds, human_move): #strat = strategy
        mapping = {
            "Always Cooperate": AlwaysCooperate,
            "Always Betray": AlwaysBetray,
            "Tit-for-Tat": TitForTat,
            "Random": lambda n: type('R', (Player,), {'make_move': lambda s, o: random.choice(["Cooperate", "Betray"])})(n)
        }
        
        if strat1 == "Human":
            p1 = Player("Human")
        else:
            p1 = mapping[strat1]("Player A")
            
        p2 = mapping[strat2]("Player B")
        
        log = ""

        for i in range(rounds): # Minimum 5 rounds
            move1 = human_move if strat1 == "Human" else p1.make_move(p2.history)
            move2 = p2.make_move(p1.history)
            
            s1, s2 = self.get_scores(move1, move2)
            p1.record_move(move1, s1)
            p2.record_move(move2, s2)
            log += f"Round {i+1}: A({move1}) vs B({move2}) | Total: A={p1.score}, B={p2.score}\n"

        winner = "Draw!"
        if p1.score > p2.score: 
            winner = "Player A Wins!"
        elif p2.score > p1.score: 
            winner = "Player B Wins!"
        
        plt.figure(figsize=(6, 4))
        plt.plot(p1.score_progression, marker='o', label=f"Player A ({strat1})")
        plt.plot(p2.score_progression, marker='s', label=f"Player B ({strat2})")
        plt.title("Prisoner's Dilemma Score Progression")
        plt.xlabel("Rounds")
        plt.ylabel("Cumulative Score")
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
         
        buf = io.BytesIO()
        plt.savefig(buf, format='png')
        plt.close()
        img = Image.open(buf)
        
        return log + f"\nFinal Result: {winner}", img

In [ ]:
engine = SimulationEngine()
choices = ["Always Cooperate", "Always Betray", "Tit-for-Tat", "Random", "Human"]

app = gr.Interface(
    fn=engine.run_sim,
    inputs=[
        gr.Dropdown(choices, label="Player A Strategy", value="Always Cooperate"),
        gr.Dropdown(choices[:-1], label="Player B AI Strategy", value="Random"),
        gr.Slider(1, 20, value=5, step=1, label="Number of Rounds"),
        gr.Radio(["Cooperate", "Betray"], label="Your Move (If Player A is 'Human')", value="Cooperate")
    ],
    outputs=[
        gr.Textbox(label="Simulation Log", lines=10), 
        gr.Image(label="Score Progression Chart")
    ],
    title="Prisoner's Dilemma Strategy Simulator",
    description="Analyze cooperation vs. betrayal using OOP and visualization."
)

if __name__ == "__main__":
    app.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio/flagged/dataset1.csv
